# 02 - Feature Engineering: BM25 + dense (TF-IDF/SVD) + reranker features

We build the three retrieval feature stacks the production system uses:
1. **BM25** sparse scores via `rank_bm25`.
2. **Dense** scores via `TF-IDF + TruncatedSVD` (a Dataiku-friendly LSA-style encoder).
3. **Reranker features** per (query, candidate) pair: ranks, scores, length ratio, domain-token match.

This notebook does **not** call any external embedding API.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from enterprise_rag.features import (
    build_bm25, build_dense_encoder, candidate_features,
    expand_sources, hybrid_score, tokenize,
)

sns.set_theme(context="notebook", style="whitegrid")

In [ ]:
corpus = pd.read_parquet("../data/processed/policy_corpus.parquet").reset_index(drop=True)
qa = pd.read_parquet("../data/processed/policy_qa.parquet").reset_index(drop=True)
print(corpus.shape, qa.shape)

## 1. BM25 index

In [ ]:
texts = corpus["text"].tolist()
bm25 = build_bm25(texts)
print("avgdl =", bm25.avgdl, "  | N docs =", len(texts))

## 2. Dense encoder (TF-IDF + SVD)

In [ ]:
enc = build_dense_encoder(texts, n_components=256)
dense_mat = enc.transform(texts)
print("dense matrix shape:", dense_mat.shape)

## 3. Score a sample query both ways

In [ ]:
sample = qa.iloc[0]
q = sample["question"]
print("Q:", q)
print("GT sources:", sample["source_doc_ids"])

bm25_s = np.asarray(bm25.get_scores(tokenize(q)))
dense_s = (dense_mat @ enc.transform([q]).T).ravel()
print("top-5 BM25 indices:", np.argsort(-bm25_s)[:5])
print("top-5 dense indices:", np.argsort(-dense_s)[:5])

## 4. Hybrid blend

In [ ]:
hybrid = hybrid_score(bm25_s, dense_s, alpha=0.5)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(hybrid, bins=40, ax=ax, color="#3b82f6")
ax.set_title(f"Hybrid score distribution for: {q[:60]}")
plt.tight_layout()
plt.show()

## 5. Reranker features for one query's top-20

In [ ]:
order = np.argsort(-hybrid)[:20]
cand = corpus.iloc[order].reset_index(drop=True)
median_len = float(np.median(corpus["text"].str.split().apply(len)))
feats = candidate_features(q, cand, bm25_s[order], dense_s[order], median_len)
feats.head(10)

## 6. Reranker feature distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(feats["bm25_score"], bins=15, ax=axes[0], color="#10b981")
axes[0].set_title("BM25 score (top-20)")
sns.histplot(feats["dense_score"], bins=15, ax=axes[1], color="#8b5cf6")
axes[1].set_title("Dense score (top-20)")
plt.tight_layout()
plt.show()

## 7. Where does the gold source land in BM25 vs. dense?

In [ ]:
key = corpus.set_index(["doc_id", "paragraph_id"])
def gold_rank(row, scores):
    pieces = expand_sources(row["source_doc_ids"])
    if not pieces:
        return np.nan
    d, p = pieces[0]
    if (d, p) not in key.index:
        return np.nan
    target_idx = key.index.get_loc((d, p))
    return int(np.where(np.argsort(-scores) == target_idx)[0][0]) + 1

ranks_bm25 = []
ranks_dense = []
for _, row in qa.head(200).iterrows():
    qq = row["question"]
    bs = np.asarray(bm25.get_scores(tokenize(qq)))
    ds = (dense_mat @ enc.transform([qq]).T).ravel()
    ranks_bm25.append(gold_rank(row, bs))
    ranks_dense.append(gold_rank(row, ds))
ranks = pd.DataFrame({"bm25": ranks_bm25, "dense": ranks_dense}).dropna()
ranks.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ranks.clip(upper=50).plot.hist(bins=25, alpha=0.55, ax=ax, color=["#10b981", "#8b5cf6"])
ax.set_xlabel("Rank of gold source (clipped at 50)")
ax.set_title("BM25 vs. dense: where the gold source lands")
plt.tight_layout()
plt.show()

## 8. Takeaway
BM25 and dense agree on the easy questions and **disagree on the rephrased ones** — exactly where the hybrid blend earns its keep. The reranker features capture the disagreement explicitly so the LR can learn the interaction.